# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pankaj1281/flyrank_ml_internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The action queue prioritizes content predicted as declining and uses observed search signals to provide a reason code. Recommendations are intended for human review and decision support, not automatic publishing or deletion.

In [17]:
from sklearn.model_selection import GroupShuffleSplit

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Shared clients:",
      len(set(train_df["client_id"]) &
          set(test_df["client_id"])))

Training rows: 23837
Test rows: 6163
Shared clients: 0


In [18]:
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

X_train = train_df[numeric_features + categorical_features]
y_train = train_df["is_declining_label"]

X_test = test_df[numeric_features + categorical_features]
y_test = test_df["is_declining_label"]

model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

model_pipeline.fit(X_train, y_train)

print("Model ready.")

Model ready.


In [20]:
import pandas as pd
import numpy as np

# Start from the held-out/test data and model predictions
action_queue = test_df.copy()

action_queue["decline_probability"] = model_pipeline.predict_proba(
    X_test
)[:, 1]

def reason_code(row):
    reasons = []

    if row["days_since_last_update"] >= 90:
        reasons.append("STALE_CONTENT")

    if row["ctr"] <= 0.1:
        reasons.append("LOW_CTR")

    if row["impressions_90d"] >= df["impressions_90d"].quantile(0.75):
        reasons.append("HIGH_IMPRESSIONS")

    if row["avg_position"] > 20:
        reasons.append("LOW_POSITION")

    return ";".join(reasons) if reasons else "MODEL_SIGNAL"

action_queue["reason_code"] = action_queue.apply(
    reason_code,
    axis=1
)

action_queue["action"] = np.where(
    action_queue["decline_probability"] >= 0.70,
    "Refresh Immediately",
    np.where(
        action_queue["decline_probability"] >= 0.50,
        "Review",
        "Monitor"
    )
)

action_queue = action_queue.sort_values(
    "decline_probability",
    ascending=False
)

action_queue["rank"] = range(1, len(action_queue) + 1)

print(action_queue[
    [
        "rank",
        "content_id",
        "decline_probability",
        "action",
        "reason_code"
    ]
].head(20))

       rank            content_id  decline_probability               action  \
5410      1  content_53285c7434ac                0.980  Refresh Immediately   
29681     2  content_9e8671965fff                0.980  Refresh Immediately   
18822     3  content_b23d650634c6                0.975  Refresh Immediately   
2139      4  content_41538bdb1b1e                0.975  Refresh Immediately   
3329      5  content_eb3b2c3bbc34                0.970  Refresh Immediately   
2043      6  content_769c5991e2f6                0.970  Refresh Immediately   
18949     7  content_7cd9b5db3ea1                0.970  Refresh Immediately   
17707     8  content_f6bf66378677                0.970  Refresh Immediately   
14343     9  content_9ac61c04930e                0.965  Refresh Immediately   
23480    10  content_9234f5075e7a                0.965  Refresh Immediately   
25063    11  content_29884c0f9255                0.965  Refresh Immediately   
17029    12  content_a4e31d98e83b                0.9

## 2. Intended use and limits

The playbook is intended for content teams to prioritize pages for review, refresh, or monitoring. It is decision-support based on observed search and content signals. It should not be used as proof that a refresh will improve traffic, and it should not automatically publish, delete, or rewrite content.

In [21]:
print("Intended users: content and SEO reviewers")
print("Purpose: prioritize content for human review")
print("Automatic publishing/deletion: not allowed")

Intended users: content and SEO reviewers
Purpose: prioritize content for human review
Automatic publishing/deletion: not allowed


## 3. Human review + the no-go list

Before acting on a recommendation, a reviewer should check the page's search position, CTR, impressions, freshness, content quality, and possible seasonal effects. Recommendations should be reviewed in context before any change is made.

The system should never automatically delete, publish, rewrite, or make claims about causal traffic impact.

In [22]:
no_go_actions = [
    "automatic deletion",
    "automatic publishing",
    "automatic rewriting",
    "claiming causal traffic impact"
]

for item in no_go_actions:
    print("NO-GO:", item)

NO-GO: automatic deletion
NO-GO: automatic publishing
NO-GO: automatic rewriting
NO-GO: claiming causal traffic impact


## 4. Monitoring / retrain triggers

The recommendations should be reviewed when the data distribution changes, prediction quality falls, or the content mix changes substantially. Retraining should be considered when new labeled outcomes become available or when measured performance on a held-out evaluation set declines.

In [23]:
print("Monitor:")
print("- model F1 on a held-out evaluation set")
print("- feature distributions")
print("- prediction distribution")
print("- content-type mix")
print("- new labeled outcomes")

print("\nRetrain when:")
print("- measured performance declines")
print("- data distribution changes materially")
print("- sufficient new labeled data becomes available")

Monitor:
- model F1 on a held-out evaluation set
- feature distributions
- prediction distribution
- content-type mix
- new labeled outcomes

Retrain when:
- measured performance declines
- data distribution changes materially
- sufficient new labeled data becomes available


## 5. Exports for the paper

The ranked action queue is exported to the work/outputs directory so it can be reused in the final research paper.

In [24]:
import os

os.makedirs("/content/flyrank_ml_internship/work/outputs", exist_ok=True)

output_path = (
    "/content/flyrank_ml_internship/work/outputs/"
    "content_action_playbook.csv"
)

action_queue[
    [
        "rank",
        "content_id",
        "decline_probability",
        "action",
        "reason_code"
    ]
].to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows exported:", len(action_queue))

Saved: /content/flyrank_ml_internship/work/outputs/content_action_playbook.csv
Rows exported: 6163


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.